# StreamFlix — Phase 1: Data Loading, Cleaning & Quality Report
**Analyst:** Vaishali Sandipsing Girase

**Goal:** Profile all 6 tables, check data quality (nulls, duplicates, types, outliers, referential integrity) and run the specific checks listed in the project brief.

## 1. Load all tables

In [23]:
import pandas as pd, numpy as np
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

sub = pd.read_csv('subscribers.csv', parse_dates=['signup_date', 'churn_date'])
tit = pd.read_csv('titles.csv', parse_dates=['date_added', 'license_expiry'])
wh  = pd.read_csv('watch_history.csv', parse_dates=['watch_date'])
rat = pd.read_csv('ratings.csv', parse_dates=['rating_date'])
rev = pd.read_csv('reviews.csv', parse_dates=['review_date'])
wl  = pd.read_csv('watchlist.csv', parse_dates=['added_date'])

tables = {'subscribers': sub, 'titles': tit, 'watch_history': wh,
          'ratings': rat, 'reviews': rev, 'watchlist': wl}
print('Total records:', sum(len(d) for d in tables.values()))

Total records: 979000


## 2. Basic profile — rows, columns, data types

In [24]:
profile = pd.DataFrame({n: {'rows': d.shape[0], 'columns': d.shape[1],
                            'full-row duplicates': int(d.duplicated().sum())}
                       for n, d in tables.items()}).T
print(profile)
print()
for n, d in tables.items():
    print(n, '->', {c: str(t) for c, t in d.dtypes.items() if 'datetime' in str(t)})

                 rows  columns  full-row duplicates
subscribers     15000       14                    0
titles           9000       21                    0
watch_history  650000       10                    0
ratings        130000        5                    0
reviews        110000        7                    0
watchlist       65000        5                    0

subscribers -> {'signup_date': 'datetime64[us]', 'churn_date': 'datetime64[us]'}
titles -> {'date_added': 'datetime64[us]', 'license_expiry': 'datetime64[us]'}
watch_history -> {'watch_date': 'datetime64[us]'}
ratings -> {'rating_date': 'datetime64[us]'}
reviews -> {'review_date': 'datetime64[us]'}
watchlist -> {'added_date': 'datetime64[us]'}


## 3. Missing values

In [25]:
rows = []
for n, d in tables.items():
    for c, v in d.isna().sum().items():
        if v > 0:
            rows.append((n, c, v, round(v / len(d) * 100, 1)))
print(pd.DataFrame(rows, columns=['table', 'column', 'null_count', 'null_%']))
print()
print('license_expiry nulls by license_type:')
print(tit.groupby('license_type').license_expiry.apply(lambda s: s.isna().sum()))

         table          column  null_count  null_%
0  subscribers      churn_date       11199    74.7
1       titles  license_expiry        1966    21.8

license_expiry nulls by license_type:
license_type
Co-Production               0
Exclusive License           0
Non-Exclusive License       0
Original                 1966
Name: license_expiry, dtype: int64


## 4. Duplicate checks

In [26]:
print('Duplicate subscriber_id :', sub.subscriber_id.duplicated().sum())
print('Duplicate title_id      :', tit.title_id.duplicated().sum())
print('Duplicate watch_id      :', wh.watch_id.duplicated().sum())

# Same subscriber rating / saving the same title more than once
d_rat = rat[rat.duplicated(['subscriber_id', 'title_id'], keep=False)]
d_wl  = wl[wl.duplicated(['subscriber_id', 'title_id'], keep=False)]
print('\nratings: rows involved in duplicate (subscriber, title):', len(d_rat),
      '| pairs with conflicting rating values:',
      int(d_rat.groupby(['subscriber_id', 'title_id']).rating.nunique().gt(1).sum()))
print('watchlist: rows involved in duplicate (subscriber, title):', len(d_wl),
      '| pairs with conflicting watched flag:',
      int(d_wl.groupby(['subscriber_id', 'title_id']).watched.nunique().gt(1).sum()))

Duplicate subscriber_id : 0
Duplicate title_id      : 0
Duplicate watch_id      : 0

ratings: rows involved in duplicate (subscriber, title): 399 | pairs with conflicting rating values: 123
watchlist: rows involved in duplicate (subscriber, title): 122 | pairs with conflicting watched flag: 31


## 5. subscribers.csv — specific checks

In [27]:
churned = sub[~sub.is_active]
print('Active:', sub.is_active.sum(), '| Churned:', len(churned),
      f'| Churn rate: {len(churned)/len(sub)*100:.1f}%')
print('Churned rows where churn_date <= signup_date :', (churned.churn_date <= churned.signup_date).sum())
print('Churned rows with missing churn_date         :', churned.churn_date.isna().sum())
print('Active rows that still have a churn_date     :', (sub.is_active & sub.churn_date.notna()).sum())
print('Age range:', sub.age.min(), '-', sub.age.max())
print('Active with tenure <= 1 month (just a note)  :', (sub.is_active & (sub.tenure_months <= 1)).sum())
print()
print(sub.groupby('plan_type').monthly_price_usd.agg(['unique', 'count']))

# Is tenure_months consistent with signup / churn dates?
ref = pd.Timestamp('2026-06-30')
end = sub.churn_date.fillna(ref)
calc = (end.dt.year - sub.signup_date.dt.year) * 12 + (end.dt.month - sub.signup_date.dt.month)
diff = (sub.tenure_months - calc)
print('\nTenure vs dates (months) — active  : mean diff', round(diff[sub.is_active].mean(), 2))
print('Tenure vs dates (months) — churned : mean diff', round(diff[~sub.is_active].mean(), 2),
      '(tenure looks measured up to today, not up to churn date)')

Active: 11199 | Churned: 3801 | Churn rate: 25.3%
Churned rows where churn_date <= signup_date : 0
Churned rows with missing churn_date         : 0
Active rows that still have a churn_date     : 0
Age range: 13 - 85
Active with tenure <= 1 month (just a note)  : 321

                 unique  count
plan_type                     
Basic with Ads   [6.99]   4458
Premium         [22.99]   4442
Standard        [15.49]   6100

Tenure vs dates (months) — active  : mean diff -0.32
Tenure vs dates (months) — churned : mean diff 23.81 (tenure looks measured up to today, not up to churn date)


## 6. watch_history.csv — specific checks

In [28]:
print('Date range:', wh.watch_date.min().date(), 'to', wh.watch_date.max().date())
print('Sessions longer than the content itself :', (wh.watch_duration_min > wh.content_duration_min).sum())
print('Zero / negative durations               :', (wh.watch_duration_min <= 0).sum())

calc_pct = wh.watch_duration_min / wh.content_duration_min * 100
print('completion_pct differs from duration/content by > 2 pts:', ((calc_pct - wh.completion_pct).abs() > 2).sum())
print('completion_pct outside 0-100                           :', ((wh.completion_pct < 0) | (wh.completion_pct > 100)).sum())
print("'completed' flag rule -> min pct when True:", wh[wh.completed].completion_pct.min(),
      '| max pct when False:', wh[~wh.completed].completion_pct.max(), '(threshold = 90%)')

m = wh.merge(tit[['title_id', 'content_duration_min']].rename(columns={'content_duration_min': 'title_dur'}), on='title_id')
print('content_duration_min mismatch vs titles table:', (m.content_duration_min != m.title_dur).sum())

print('\nwatch_duration_min summary:')
print(wh.watch_duration_min.describe().round(1))
print('\nSessions by device:')
print(wh.device.value_counts())

Date range: 2017-03-10 to 2026-05-31
Sessions longer than the content itself : 0
Zero / negative durations               : 0
completion_pct differs from duration/content by > 2 pts: 0
completion_pct outside 0-100                           : 0
'completed' flag rule -> min pct when True: 90.0 | max pct when False: 90.0 (threshold = 90%)
content_duration_min mismatch vs titles table: 0

watch_duration_min summary:
count    650000.0
mean        307.7
std         428.4
min           0.7
25%          63.7
50%          98.4
75%         405.6
max        5701.8
Name: watch_duration_min, dtype: float64

Sessions by device:
device
Smart TV           221446
Mobile             175605
Laptop              96408
Tablet              65276
Game Console        52322
Streaming Stick     38943
Name: count, dtype: int64


### Outlier check — long sessions
Some sessions are very long (max ≈ 5,700 min ≈ 95 hrs). They are **not** errors, because each is shorter than the content itself (TV shows have a combined runtime across all seasons). Still, it is useful to flag them.

In [29]:
q99 = wh.watch_duration_min.quantile(0.99)
print('99th percentile of watch_duration_min:', round(q99, 1))
print('Sessions above 99th percentile:', (wh.watch_duration_min > q99).sum())
print('\nAverage session length by content type (min):')
print(wh.merge(tit[['title_id', 'type']], on='title_id').groupby('type').watch_duration_min.agg(['mean', 'median', 'max']).round(1))

99th percentile of watch_duration_min: 1968.7
Sessions above 99th percentile: 6500

Average session length by content type (min):
          mean  median     max
type                          
Movie     70.8    70.6   182.5
TV Show  646.2   508.4  5701.8


## 7. Referential integrity

In [30]:
for name, d in [('watch_history', wh), ('ratings', rat), ('reviews', rev), ('watchlist', wl)]:
    print(f'{name:14s} subscriber_id missing in subscribers: {(~d.subscriber_id.isin(sub.subscriber_id)).sum()} | '
          f'title_id missing in titles: {(~d.title_id.isin(tit.title_id)).sum()}')

print('\nSubscribers with zero sessions:', (~sub.subscriber_id.isin(wh.subscriber_id)).sum())
print('Titles never watched          :', (~tit.title_id.isin(wh.title_id)).sum())
print('\ntitles.total_plays sum vs watch_history rows :', tit.total_plays.sum(), 'vs', len(wh))
print('titles.total_watch_hours vs sessions (hours) :', round(tit.total_watch_hours.sum(), 1),
      'vs', round(wh.watch_duration_min.sum() / 60, 1))

watch_history  subscriber_id missing in subscribers: 0 | title_id missing in titles: 0
ratings        subscriber_id missing in subscribers: 0 | title_id missing in titles: 0
reviews        subscriber_id missing in subscribers: 0 | title_id missing in titles: 0
watchlist      subscriber_id missing in subscribers: 0 | title_id missing in titles: 0

Subscribers with zero sessions: 2
Titles never watched          : 0

titles.total_plays sum vs watch_history rows : 650000 vs 650000
titles.total_watch_hours vs sessions (hours) : 3333473.0 vs 3333467.7


## 8. reviews, ratings, watchlist — value checks

In [31]:
print('Sentiment values:', rev.sentiment.value_counts(dropna=False).to_dict())
print('Rating values   :', rat.rating.value_counts().sort_index().to_dict())
print('Negative helpful_votes:', (rev.helpful_votes < 0).sum())
print('Watchlist watched=True share:', round(wl.watched.mean() * 100, 1), '%')
print('Movies with seasons != 0:', ((tit.type == 'Movie') & (tit.seasons != 0)).sum())
print('Quality score outside 0-100:', ((tit.quality_score < 0) | (tit.quality_score > 100)).sum())
print('Genres:', tit.primary_genre.nunique(), '| Languages:', tit.language.nunique(), '| Countries (subs):', sub.country.nunique())

Sentiment values: {'Positive': 75007, 'Neutral': 22759, 'Negative': 12234}
Rating values   : {1: 1786, 2: 14570, 3: 44491, 4: 48564, 5: 20589}
Negative helpful_votes: 0
Watchlist watched=True share: 46.2 %
Movies with seasons != 0: 0
Quality score outside 0-100: 0
Genres: 16 | Languages: 14 | Countries (subs): 20


## 9. Logical consistency — sessions after churn  (main finding)

In [32]:
w = wh.merge(sub[['subscriber_id', 'signup_date', 'churn_date', 'is_active']], on='subscriber_id')
print('Sessions before signup date:', (w.watch_date < w.signup_date).sum())

post = w[w.watch_date > w.churn_date]
print('Sessions AFTER churn date  :', len(post), f'({len(post)/len(wh)*100:.1f}% of all sessions)')
print('Churned subscribers involved:', post.subscriber_id.nunique(), 'of', (~sub.is_active).sum())
print('Hours in those sessions     :', round(post.watch_duration_min.sum() / 60), 'of', round(wh.watch_duration_min.sum() / 60))
print('Days after churn (median)   :', int((post.watch_date - post.churn_date).dt.days.median()))
print('Share of churned subs\' sessions that happen after churn:',
      round(len(post) / len(w[~w.is_active]) * 100, 1), '%')

# Flag instead of deleting; later phases can decide
wh_clean = wh.merge(sub[['subscriber_id', 'churn_date']], on='subscriber_id', how='left')
wh_clean['after_churn_flag'] = wh_clean.watch_date > wh_clean.churn_date
print('\nFlag column created:', wh_clean.after_churn_flag.sum(), 'rows flagged')

Sessions before signup date: 0
Sessions AFTER churn date  : 131199 (20.2% of all sessions)
Churned subscribers involved: 3708 of 3801
Hours in those sessions     : 672575 of 3333468
Days after churn (median)   : 447
Share of churned subs' sessions that happen after churn: 81.1 %

Flag column created: 131199 rows flagged


## 10. Cleaning actions (flag, don't blindly delete)

In [33]:
# 1. Duplicate (subscriber, title) ratings -> keep the latest rating
rat_clean = rat.sort_values('rating_date').drop_duplicates(['subscriber_id', 'title_id'], keep='last')
# 2. Duplicate watchlist entries -> keep the latest entry
wl_clean = wl.sort_values('added_date').drop_duplicates(['subscriber_id', 'title_id'], keep='last')
# 3. Fill license_expiry for Originals with a clear label (they never expire)
tit_clean = tit.copy()
tit_clean['license_expiry_note'] = np.where(tit_clean.license_expiry.isna(), 'No expiry (Original)', 'Has expiry')
# 4. Add a proper churned flag and corrected tenure (up to churn date or 2026-06-30)
sub_clean = sub.copy()
end = sub_clean.churn_date.fillna(pd.Timestamp('2026-06-30'))
sub_clean['tenure_months_corrected'] = (end.dt.year - sub_clean.signup_date.dt.year) * 12 + (end.dt.month - sub_clean.signup_date.dt.month)

print('ratings   :', len(rat), '->', len(rat_clean))
print('watchlist :', len(wl), '->', len(wl_clean))
print('after_churn_flag added to watch_history (rows kept):', len(wh_clean))
print('tenure_months_corrected added to subscribers')

ratings   : 130000 -> 129800
watchlist : 65000 -> 64939
after_churn_flag added to watch_history (rows kept): 650000
tenure_months_corrected added to subscribers


## 11. Data Quality Report — Summary

**Overall:** the dataset is structurally very clean. All 6 tables load correctly, with **979,000 records**, no duplicate primary keys, no broken links between tables, and no invalid dates or values. The issues below are mostly *logical* inconsistencies that an analyst must know about before drawing conclusions.

| # | Finding | Table | Size | Severity | Action |
|---|---------|-------|------|----------|--------|
| 1 | **Sessions recorded after the subscriber's churn date** | watch_history | 131,199 sessions (20.2%), 3,708 of 3,801 churned subscribers, ~672K watch-hours | **High** | Flagged with `after_churn_flag`; do not delete. Report both ways in Phase 3 |
| 2 | Duplicate (subscriber, title) ratings | ratings | 200 extra rows, 123 pairs with conflicting scores | Medium | Kept latest rating |
| 3 | Duplicate (subscriber, title) watchlist entries | watchlist | 61 extra rows, 31 pairs with conflicting `watched` | Low | Kept latest entry |
| 4 | `tenure_months` ignores churn date (measured to today for churned users) | subscribers | ~3,800 churned subs affected | Medium | Added `tenure_months_corrected` |
| 5 | Ratings given for titles the subscriber never watched | ratings | ~98.6% of ratings have no matching session | Info | Synthetic-data trait; ratings are analysed separately from watch data |
| 6 | `license_expiry` blank for all Originals | titles | 1,966 (21.8%) | Info (expected) | Labelled "No expiry (Original)" |
| 7 | 2 subscribers have no sessions | subscribers | 2 | Info | Keep |
| 8 | Very long sessions (up to ~95 hrs) | watch_history | top 1% | Info | Not errors; TV Show runtimes cover all seasons |

**Checks that passed:** unique `watch_id` / `subscriber_id` / `title_id`; every `subscriber_id` and `title_id` in child tables exists in parent tables; no churn_date ≤ signup_date; no active subscriber with a churn_date; no session longer than its content; `completion_pct` matches duration ÷ content length; `completed` = 90%+ completion; sentiment only Positive / Neutral / Negative; ratings only 1–5; prices match plans.

**Why the churn finding matters:** if 20% of viewing happens after a user has "churned", then either churn dates are wrong or some viewing is on shared / reactivated accounts. In Phase 3 (churn and watch-time KPIs), results should be shown *with and without* these sessions.